What Is a Gateway?
A gateway is a proxy layer between your application and any LLM API:

Your App  →  [LLM Gateway]  →  Groq / NVIDIA / OpenAI / Anthropic
The gateway intercepts every request and can:

Route to the right provider
Retry automatically on transient failures
Fallback to a different provider if the primary fails
Balance load across multiple models by weight
Cache responses so identical requests never hit the LLM twice
Log everything with full request/response detail
Tag requests with metadata (user, session, feature) for analytics
Enforce timeouts and kill slow requests

Why Portkey?
Feature	           Portkey	LiteLLM	Direct SDK
250+ models unified	✅	✅	❌ one provider
Automatic fallbacks	✅	✅	❌ manual
Request caching	✅	✅	❌ manual
Observability dashboard	✅ beautiful UI	⚠️ basic	❌ none
LangChain drop-in	✅	✅	✅ native
Config-as-code	✅ JSON/YAML	❌	❌
Open source	✅ Apache 2.0	✅ MIT	N/A
Overhead	~20–40ms	~30ms	0ms

In [33]:
import os
import time
import uuid
import json
from dotenv import load_dotenv
from portkey_ai import Portkey, createHeaders, PORTKEY_GATEWAY_URL

load_dotenv(dotenv_path="../.env")  # Load environment variables from .env file
PORTKEY_API_KEY = os.getenv("PORTKEY_API_KEY")
if not PORTKEY_API_KEY:
    raise ValueError("PORTKEY_API_KEY is missing. Set it in ../.env before running this notebook.")
PORTKEY_CONFIG_ID = "flight-policy"

GROQ_SLUG    =  "rag"  # your Groq integration slug
GROQ_MODEL   = f"@{GROQ_SLUG}/llama-3.3-70b-versatile"

# Second Groq integration for multi-provider experiments (5, 6, 10)
# Uses a smaller/faster model as the fallback target
GROQ_SLUG_2      =  "rag"           # your second Groq slug
GROQ_MODEL_SMALL = f"@{GROQ_SLUG_2}/llama-3.1-8b-instant"

# Keep GROQ_API_KEY for the LangChain experiment (Exp 9)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if PORTKEY_API_KEY and GROQ_API_KEY:
    print("✅ All API keys loaded successfully!")

✅ All API keys loaded successfully!


In [13]:
def section(title):
    print(f"\n{'='*62}")
    print(f"  {title}")
    print(f"{'='*62}")

def show(q, answer, ms, label=""):
    bar = chr(9472) * 62
    print(f"\n{bar}")
    print(f"Q: {q}")
    print(f"A: {answer[:260]}{'...' if len(answer) > 260 else ''}")
    note = f" | {label}" if label else ""
    print(f"⏱  {ms:.0f}ms{note}")
    print(bar)

# Main Portkey client — used for all simple experiments
portkey = Portkey(api_key=PORTKEY_API_KEY)

print("Setup complete!")
print(f"  Portkey API Key : {'OK' if PORTKEY_API_KEY else 'MISSING'}")
print(f"  Groq slug       : {GROQ_SLUG}")
print(f"  Groq model ref  : {GROQ_MODEL}")
print(f"  Groq slug 2     : {GROQ_SLUG_2}")
print(f"  Small model ref : {GROQ_MODEL_SMALL}")
print(f"\nPortkey Gateway : {PORTKEY_GATEWAY_URL}")

Setup complete!
  Portkey API Key : OK
  Groq slug       : rag
  Groq model ref  : @rag/llama-3.3-70b-versatile
  Groq slug 2     : rag
  Small model ref : @rag/llama-3.1-8b-instant

Portkey Gateway : https://api.portkey.ai/v1


In [14]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
raw_groq = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)
section("BASELINE — Direct Groq Call")
questions = [
    "What is Kubernetes in one sentence?",
    "What is Intel SRIOV?",
]
for q in questions:
    start = time.time()
    r = raw_groq.invoke([HumanMessage(content=q)])
    show(q, r.content, (time.time() - start) * 1000, label="Direct Groq Call")


  BASELINE — Direct Groq Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open-source container orchestration system that automates the deployment, scaling, and management of containerized applications across a cluster of machines.
⏱  330ms | Direct Groq Call
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that allows a single physical device, such as a network interface card (NIC) or a storage controller, to appear as multiple virtual devices to the operating system and applications. This is achieved...
⏱  1765ms | Direct Groq Call
──────────────────────────────────────────────────────────────


In [15]:
section("EXP 1 — Basic Gateway Call")
for q in questions:
    start = time.time()
    response = portkey.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": q}],)
    show(q, response.choices[0].message.content, (time.time() - start) * 1000, label="Basic Gateway Call")
print("\n✅ Check portkey.ai → Logs to see both requests fully logged!")
print("   Token count, cost, latency — all tracked. Zero extra code.")


  EXP 1 — Basic Gateway Call

──────────────────────────────────────────────────────────────
Q: What is Kubernetes in one sentence?
A: Kubernetes is an open-source container orchestration system that automates the deployment, scaling, and management of containerized applications across a cluster of machines.
⏱  876ms | Basic Gateway Call
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: What is Intel SRIOV?
A: Intel SR-IOV (Single Root I/O Virtualization) is a technology that allows multiple virtual machines (VMs) to share a single physical device, such as a network interface card (NIC) or a graphics card, in a virtualized environment. It's a standard developed by t...
⏱  1732ms | Basic Gateway Call
──────────────────────────────────────────────────────────────

✅ Check portkey.ai → Logs to see both requests fully logged!
   Token count, cost, latency — all tracked. Zero extra code.


# Experiment2 metaData & Observability

In [19]:
session = str(uuid.uuid4())[:8]

scenarios = [
    ("alice", "enterprise-rag",   "What is Kubernetes RBAC?"),
    ("bob",   "docs-chatbot",     "How does BGP path selection work?"),
    ("carol", "support-bot",      "What is SRIOV virtualization?"),
    ("alice", "enterprise-rag",   "Explain Kubernetes NetworkPolicy"),   #
]

for user, feature, q in scenarios:
    t0 = time.time()
    response = portkey.with_options(
        metadata={
            "_user": user, 
            "feature": feature, 
            "session_id": session,
            "environment": "notebook"}
    ).chat.completions.create(model=GROQ_MODEL, messages=[{"role": "user", "content": q}])
    ms = (time.time() - t0) * 1000
    print(f"\n👤 {user:8s} | 🔧 {feature:18s} | {ms:.0f}ms")
    print(f"  Q: {q}")
    print(f"  A: {response.choices[0].message.content[:120]}...")

print("\n✅ Dashboard now shows: cost per user, calls per feature, session grouping")
    


👤 alice    | 🔧 enterprise-rag     | 1788ms
  Q: What is Kubernetes RBAC?
  A: Kubernetes Role-Based Access Control (RBAC) is a security mechanism that enables you to control access to Kubernetes res...

👤 bob      | 🔧 docs-chatbot       | 2767ms
  Q: How does BGP path selection work?
  A: BGP (Border Gateway Protocol) path selection is a complex process that involves evaluating multiple factors to determine...

👤 carol    | 🔧 support-bot        | 1940ms
  Q: What is SRIOV virtualization?
  A: SR-IOV (Single-Root Input/Output Virtualization) is a technology that enables multiple virtual machines (VMs) to share a...

👤 alice    | 🔧 enterprise-rag     | 2558ms
  Q: Explain Kubernetes NetworkPolicy
  A: **Kubernetes NetworkPolicy**

Kubernetes NetworkPolicy is a network segmentation feature that...

✅ Dashboard now shows: cost per user, calls per feature, session grouping


# EXPERIMENT 3 -- Automatic Retries

In [21]:
retry_config = {
    "retry":{
        "attempts": 3,
        "on_status": [429, 500, 502, 503, 504]
    }
}

portkey_retry = Portkey(api_key=PORTKEY_API_KEY, retry_config=retry_config)
section("EXP 3 — Automatic Retries")
print("Config: 3 retry attempts on [429, 500, 502, 503, 504]")
print("Retries fire automatically on failure — transparent to your code\n")

try:
    t0 = time.time()
    response = portkey_retry.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is Kubernetes?"}] )
    ms = (time.time() - t0) * 1000
    print(f"✅ Success after retries! Time: {ms:.0f}ms")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\nRetry sequence if Groq had failed:")
    print("  Attempt 1 → 429 → wait 1s → Attempt 2 → 429 → wait 2s → Attempt 3")
    print("  Your code only sees the final success or the last failure")
except Exception as e:
    print(f"❌ Failed after retries: {e}")



  EXP 3 — Automatic Retries
Config: 3 retry attempts on [429, 500, 502, 503, 504]
Retries fire automatically on failure — transparent to your code

✅ Success after retries! Time: 1545ms
❌ Failed after retries: 'AIMessage' object has no attribute 'choices'


# Experiment4 -- Request timeouts

In [27]:
timeout_config = {"request_timeout": 10000}   # 10 seconds in ms
portkey_timeout = Portkey(api_key=PORTKEY_API_KEY, timeout_config=timeout_config)

section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

section("EXP 4 — Request Timeouts")
print("Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.\n")

try:
    t0 = time.time()
    r = portkey_timeout.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Explain Kubernetes networking in 2 sentences."}],
        wait_for_output=15000)
    ms = (time.time() - t0) * 1000
    print(f"✅ Response in {ms:.0f}ms (within 10s timeout)")
    print(f"   {r.choices[0].message.content}")
except Exception as e:
    print(f"⏱  Timed out: {e}")
    print("   Portkey issued a 408. Pair with fallback to auto-switch providers on timeout.")

print("\n--- Combining timeout + retry (production pattern) ---")
combined = {
    "request_timeout": 10000,
    "retry": {"attempts": 2, "on_status_codes": [408, 429, 503]}
}
print(json.dumps(combined, indent=1))


  EXP 4 — Request Timeouts
Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.


  EXP 4 — Request Timeouts
Timeout: 10,000ms (10 seconds). Portkey returns HTTP 408 if exceeded.

✅ Response in 884ms (within 10s timeout)
   Kubernetes networking provides a way for pods to communicate with each other and with services outside the cluster, using a combination of virtual networks, network policies, and service discovery mechanisms to manage traffic flow and accessibility. The Kubernetes networking model includes components such as pods, services, ingress controllers, and network policies, which work together to enable secure and scalable communication between applications and services within and outside the cluster.

--- Combining timeout + retry (production pattern) ---
{
 "request_timeout": 10000,
 "retry": {
  "attempts": 2,
  "on_status_codes": [
   408,
   429,
   503
  ]
 }
}


# Experiment5 -- Fallback

In [28]:
fallback_config = {
    "strategy": {"mode": "fallback"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}},     # primary
        {"override_params": {"model": GROQ_MODEL_SMALL}}    # fallback if primary fails
    ]
}
portkey_fallback = Portkey(api_key=PORTKEY_API_KEY, config=fallback_config)
section("EXP 5 — Fallback Routing")
print(f"Strategy: {GROQ_MODEL} → {GROQ_MODEL_SMALL} on failure\n")

fallback_questions = [
    "What is Intel QuickAssist Technology?",
    "Explain Kubernetes persistent volume claims.",
]

for q in fallback_questions:
    try:
        t0 = time.time()
        r = portkey_fallback.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label="fallback ready")
    except Exception as e:
        print(f"❌ {e}")
        print("   → Check that both GROQ_SLUG and GROQ_SLUG_2 are set correctly in c04")

print("\nFallback chain:")
print("  Groq 2xx   → return immediately")
print("  Groq 4xx/5xx → try small Groq model (8b) automatically")
print("  Check Portkey Logs to see fallback activations")


  EXP 5 — Fallback Routing
Strategy: @rag/llama-3.3-70b-versatile → @rag/llama-3.1-8b-instant on failure


──────────────────────────────────────────────────────────────
Q: What is Intel QuickAssist Technology?
A: Intel QuickAssist Technology (Intel QAT) is a set of hardware and software components designed to accelerate specific workloads, particularly those related to security, compression, and data processing. It is primarily targeted at data centers, cloud infrastru...
⏱  1697ms | fallback ready
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: Explain Kubernetes persistent volume claims.
A: **Kubernetes Persistent Volume Claims (PVCs)**

In Kubernetes, a Persistent Volume Claim (PVC) is a request for storage resources that can be used by a pod. It's a way to dynamically provision storage for y...
⏱  1887ms | fallback ready
──────────────────────────────────────────────────────────────

Fallback chain:

# Experiment 6 LoadBalancing

In [35]:
load_balancing_config = {
    "strategy": {"mode": "loadbalance"},
    "targets": [
        {"override_params": {"model": GROQ_MODEL}, "weight": 0.7},
        {"override_params": {"model": GROQ_MODEL_SMALL}, "weight": 0.3}
    ]
}

portkey_lob = Portkey(api_key=PORTKEY_API_KEY, config=load_balancing_config)

section("EXP 6 — Load Balancing (70% large / 30% small)")


lb_questions = [
    "What is a Kubernetes Ingress resource?",
    "How does OSPF differ from BGP?",
    "What is Intel FPGA acceleration?",
    "Explain Kubernetes HPA.",
    "What is a VLAN trunk?",
    "How does Kubernetes etcd work?",
]

print("Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).\n")


for i, q in enumerate(lb_questions, 1):
    try:
        t0 = time.time()
        r = portkey_lob.chat.completions.create(
            messages=[{"role": "user", "content": q}]
        )
        ms = (time.time() - t0) * 1000
        show(q, r.choices[0].message.content, ms, label=f"LB request {i}")
    except Exception as e:
        print(f"Error on request {i}: {e}")


  EXP 6 — Load Balancing (70% large / 30% small)
Sending 6 requests. Expect ~4 on large model (70b), ~2 on small model (8b) (probabilistic).


──────────────────────────────────────────────────────────────
Q: What is a Kubernetes Ingress resource?
A: A Kubernetes Ingress resource is a Kubernetes object that provides load balancing, path-based routing, and SSL/TLS termination for incoming HTTP requests to a Kubernetes cluster. It acts as an entry point for external traffic to access services within the clus...
⏱  1891ms | LB request 1
──────────────────────────────────────────────────────────────

──────────────────────────────────────────────────────────────
Q: How does OSPF differ from BGP?
A: OSPF (Open Shortest Path First) and BGP (Border Gateway Protocol) are two of the most widely used routing protocols in the Internet. Here's a comparison of their key differences:

**1. Purpose:**
	* OSPF: Designed for large-scale interior routing within a sing...
⏱  1229ms | LB request 2
──────

# Experiment 7 Cache

In [37]:
cache_config = {"cache": {"mode": "simple"}}
portkey_cache = Portkey(api_key=PORTKEY_API_KEY, config=cache_config)

section("EXP 7 — Simple Caching")


q = "Define Kubernetes ConfigMap in one sentence."

call_params = dict(model=GROQ_MODEL, 
                messages=[{"role": "user", "content": q}],
                temperature=0,
                max_tokens=200)

print("--- CALL 1: Cache MISS — Portkey forwards to Groq ---")

t0 = time.time()
r1 = portkey_cache.chat.completions.create(**call_params)
t1 = time.time()
print(f"--- CALL 1 DURATION: {t1 - t0:.2f} seconds ---")
print(f"Answer is : {r1.choices[0].message.content[:200]}...")

print("--- CALL 2: Cache HIT — Portkey returns cached response ---")

t0 = time.time()
r2 = portkey_cache.chat.completions.create(**call_params)
t1 = time.time()
print(f"--- CALL 2 DURATION: {t1 - t0:.2f} seconds ---")
print(f"Answer is : {r2.choices[0].message.content[:200]}...")


  EXP 7 — Simple Caching
--- CALL 1: Cache MISS — Portkey forwards to Groq ---
--- CALL 1 DURATION: 0.42 seconds ---
Answer is : A Kubernetes ConfigMap is a resource that stores and manages configuration data, such as environment variables, configuration files, and other settings, for applications running in a Kubernetes cluste...
--- CALL 2: Cache HIT — Portkey returns cached response ---
--- CALL 2 DURATION: 0.06 seconds ---
Answer is : A Kubernetes ConfigMap is a resource that stores and manages configuration data, such as environment variables, configuration files, and other settings, for applications running in a Kubernetes cluste...


# Save for config Dashboard

In [39]:
PORTKEY_CONFIG_ID = os.getenv("PORTKEY_SAVED_CONFIG_ID", "pc-pc-121-395c02")

In [42]:
portkey_from_config = Portkey(
    api_key=PORTKEY_API_KEY,
    config=PORTKEY_CONFIG_ID    # "pc-xxxxx"  ← different from provider slug!
)

section("EXP 8 — Saved Config from Dashboard")
print(f"Using config ID: {PORTKEY_CONFIG_ID}")
print("All routing rules live in the dashboard. Edit there, no redeploy needed.\n")

try:
    t0 = time.time()
    r = portkey_from_config.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "What is Kubernetes HPA?"}]
    )
    ms = (time.time() - t0) * 1000
    print(f"✅ Config applied ({ms:.0f}ms)")
    print(f"   {r.choices[0].message.content[:300]}")
    print("\n💡 To update routing: edit config in dashboard → Save. Zero redeployment.")
except Exception as e:
    print(f"❌ Config error: {e}")
    print("   → Config IDs start with pc- and come from Portkey dashboard → Configs")


  EXP 8 — Saved Config from Dashboard
Using config ID: pc-pc-121-395c02
All routing rules live in the dashboard. Edit there, no redeploy needed.

✅ Config applied (112ms)
   Kubernetes Horizontal Pod Autoscaling (HPA) is a feature in Kubernetes that automatically adjusts the number of replicas of a pod based on the observed CPU utilization or other custom metrics. This allows you to scale your application horizontally, adding or removing replicas as needed to maintain a

💡 To update routing: edit config in dashboard → Save. Zero redeployment.


# langChain Integration with PortKey

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# For OpenAI-compatible clients, explicitly pass Portkey routing headers.
# Using provider=groq satisfies Portkey's requirement for x-portkey-provider/config.
portkey_llm = ChatOpenAI(
    api_key=PORTKEY_API_KEY,
    base_url=PORTKEY_GATEWAY_URL,
    model=GROQ_MODEL,
    temperature=0,
    default_headers=createHeaders(
        api_key=PORTKEY_API_KEY,
        provider="groq",
        metadata={
            "_user": "rag-pipeline",
            "environment": "notebook",
            "feature": "langchain-integration"
        }
    )
)

section("EXP 9 — LangChain Drop-in")

# Test 1: Direct invoke (like app/agents/nodes/planner.py)
print("--- Test 1: Direct invoke (like planner node) ---")
t0 = time.time()
r = portkey_llm.invoke([
    SystemMessage(content="You are an Enterprise IT Assistant."),
    HumanMessage(content="What is the difference between a Deployment and a StatefulSet?")
])
ms = (time.time() - t0) * 1000
print(f"✅ {ms:.0f}ms")
print(r.content[:250])


  EXP 9 — LangChain Drop-in
--- Test 1: Direct invoke (like planner node) ---


BadRequestError: Error code: 400 - {'status': 'failure', 'message': 'Either x-portkey-config or x-portkey-provider header is required'}


  EXP 9 — LangChain Drop-in
--- Test 1: Direct invoke (like planner node) ---


BadRequestError: Error code: 400 - {'status': 'failure', 'message': 'Either x-portkey-config or x-portkey-provider header is required'}